# RAG-Vorlesungs-Tutor — Client

Dünner Client für das Backend. **Agent, Retrieval und LLM laufen im Server**
(`server/api.py`) und werden beim API-Start aufgebaut. Dieses Notebook lädt nur
PDFs hoch und stellt Fragen – alles über HTTP.

**Server vorher starten** (eigenes Terminal):
```bash
cd server
python api.py
```

## Setup

In [69]:
import requests
from pathlib import Path

API_URL = "http://127.0.0.1:8000"

## PDFs hochladen (`/ingest`)

Legt die PDFs aus `input_pdfs/` per Upload beim Backend ab; dort werden sie mit
docling in Markdown gewandelt, gechunkt und in die Vektor-DB geschrieben.
Der erste Upload lädt einmalig die docling-Modelle → das kann dauern.

In [70]:
PDF_DIR = Path.cwd() / "input_pdfs"
pdf_paths = sorted(PDF_DIR.glob("*.pdf"))
print(f"{len(pdf_paths)} PDF(s):", [p.name for p in pdf_paths])

if not pdf_paths:
    print("Keine PDFs gefunden – lege welche in", PDF_DIR)
else:
    handles = [open(p, "rb") for p in pdf_paths]
    try:
        files = [("files", (p.name, fh, "application/pdf"))
                 for p, fh in zip(pdf_paths, handles)]
        # formulas=true aktiviert die Formel-/LaTeX-Interpretation (langsamer).
        # Optional zusaetzlich: "ocr": "true" (gescannte PDFs), "delete_pdfs": "false".
        resp = requests.post(f"{API_URL}/ingest", files=files, data={"formulas": "true"})
        print("Status:", resp.status_code)
        print(resp.json())
    finally:
        for fh in handles:
            fh.close()

0 PDF(s): []
Keine PDFs gefunden – lege welche in /home/tim/Dokumente/NLP/rag-lecture-tutor/input_pdfs


## Frage stellen (`/ask`)

Schickt die Frage an den Backend-Agenten und zeigt dessen Antwort.

In [71]:
frage = input("Frage: ")
resp = requests.post(f"{API_URL}/ask", json={"query": frage})
if resp.status_code == 200:
    print(resp.json()["answer"])
else:
    print("Fehler:", resp.status_code, resp.text)

In der Datei **example.md** findest du ein Beispiel‑Firmenhandbuch, das folgende Abschnitte enthält:

- **Öffnungszeiten** – Das Büro ist von Montag bis Freitag von 09:00 bis 17:00 Uhr geöffnet.  
- **IT‑Support** – Passwortprobleme per E‑Mail an support@example.com, dringende Störungen telefonisch unter +49 30 12345678; Software‑Installationen bedürfen IT‑Freigabe.  
- **Urlaubsregelung** – 30 Urlaubstage pro Jahr, Antrag mindestens zwei Wochen vorher, Resturlaub bis 31. März des Folgejahres.  
- **Reisekosten** – Bahn‑2. Klasse erstattet, Hotelkosten bis 120 € pro Nacht, Taxis nur bei dringendem Anlass und mit Beleg.  
- **Projekt Alpha** – Start am 15. Juli 2026, Ansprechpartnerin Maria Schneider, wöchentliche Status‑Meetings montags um 10:00 Uhr.  
- **Sicherheitsregeln** – Besucher müssen sich anmelden, Schreibtische sperren, Zutrittskarten dürfen nicht weitergegeben werden.  
- **Arbeitszeiten** – 40 Stunden pro Woche, Überstunden benötigen Führungskraft‑Genehmigung.  
- **Homeof